# 5 — Evaluation and error analysis

Requires a trained checkpoint and its `evaluation_test.json`:

```bash
python -m nmt.evaluation.evaluate --checkpoint artifacts/checkpoints/bpe_scratch/best_bleu.pt
```


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from nmt.utils.io import project_root, read_json
from nmt.viz.style import use_style

use_style()
print("project root:", ROOT)

In [ ]:
# Prefer the primary model, but fall back to whatever has been evaluated so
# far, so the notebook is useful before the full runs have finished.
PREFERRED = "bpe_scratch"

available = sorted(
    p.parent.name
    for p in (ROOT / "artifacts" / "results").glob("*/evaluation_test.json")
)
if not available:
    raise SystemExit(
        "No evaluation results found. Train and evaluate a model first:\n"
        "  python -m nmt.training.train --config configs/bpe_scratch.yaml\n"
        "  python -m nmt.evaluation.evaluate "
        "--checkpoint artifacts/checkpoints/bpe_scratch/best_bleu.pt"
    )

RUN = PREFERRED if PREFERRED in available else available[0]
print("available runs:", available)
print("inspecting    :", RUN)

report = read_json(ROOT / "artifacts" / "results" / RUN / "evaluation_test.json")
print("checkpoint:", report["checkpoint"])
print("split     :", report["split"], f"({report['pairs']:,} pairs)")
print("decoding  :", report["decoding"])
print()
for direction, values in report["summary"]["bleu"].items():
    chrf = report["summary"]["chrf2"][direction]
    print(f"  {direction}   BLEU {values:6.2f}   chrF2 {chrf:6.2f}")

## Our BLEU against sacreBLEU

We implement BLEU from the definition as course material, but quote sacreBLEU
in the report because BLEU is famously sensitive to tokenisation. They must
agree.

In [ ]:
for direction in ("en-es", "es-en"):
    metrics = report["directions"][direction]["metrics"]
    ours = metrics["from_scratch"]["score"]
    theirs = metrics["sacrebleu"]["bleu"]
    print(f"{direction}: ours {ours:.4f}   sacrebleu {theirs:.4f}   "
          f"difference {abs(ours - theirs):.5f}")
print("\nsignature:", report["directions"]["en-es"]["metrics"]["sacrebleu"]["signature"])

## Greedy against beam search

In [ ]:
if "greedy_bleu" in report:
    import pandas as pd
    rows = []
    for direction in ("en-es", "es-en"):
        rows.append({
            "direction": direction,
            "greedy": round(report["greedy_bleu"][direction], 2),
            "beam": round(report["summary"]["bleu"][direction], 2),
            "gain": round(report["beam_gain"][direction], 2),
        })
    display(pd.DataFrame(rows))

## Quality against sentence length

The claim being tested: a recurrent model carries information across a number
of sequential steps proportional to sentence length, whereas in a transformer
every pair of positions is one attention hop apart — so the recurrent model
should degrade faster on long inputs.

In [ ]:
import pandas as pd

frames = []
for direction in ("en-es", "es-en"):
    buckets = report["directions"][direction]["error_analysis"]["bleu_by_length"]
    frame = pd.DataFrame(buckets)[["bucket", "sentences", "mean_sentence_bleu", "mean_length_ratio"]]
    frame["direction"] = direction
    frames.append(frame)
display(pd.concat(frames).round(2))

## Failure-mode census

In [ ]:
for direction in ("en-es", "es-en"):
    analysis = report["directions"][direction]["error_analysis"]
    print(f"\n=== {direction} ({analysis['sentences']:,} sentences) ===")
    for name, rate in sorted(analysis["category_rates"].items(), key=lambda x: -x[1]):
        count = analysis["category_counts"][name]
        print(f"  {name:22s} {count:6,}  ({rate:5.1%})")

## The twenty worst translations

The detectors are a reading aid, not a verdict — read the examples and judge
each one. Some `no_content_overlap` cases are legitimate paraphrases that BLEU
simply cannot see.

In [ ]:
for direction in ("en-es", "es-en"):
    print(f"\n{'=' * 78}\n{direction}\n{'=' * 78}")
    for i, example in enumerate(report["directions"][direction]["error_analysis"]["worst"][:20], 1):
        print(f"\n{i:2d}. sBLEU {example['sentence_bleu']:.1f}   [{', '.join(example['categories'])}]")
        print(f"    source : {example['source']}")
        print(f"    ref    : {example['reference']}")
        print(f"    model  : {example['hypothesis']}")

## And what it gets right

A report that only shows failures gives no sense of what the system does
well.

In [ ]:
for direction in ("en-es", "es-en"):
    print(f"\n=== {direction} ===")
    for example in report["directions"][direction]["error_analysis"]["best"][:8]:
        print(f"  {example['source']}")
        print(f"    -> {example['hypothesis']}")

## Comparing every system

In [ ]:
from nmt.viz.results_plots import load_reports, RUN_LABELS

reports = load_reports(ROOT / "artifacts" / "results")
rows = []
for name, r in reports.items():
    rows.append({
        "system": RUN_LABELS.get(name, name),
        "BLEU en-es": round(r["summary"]["bleu"]["en-es"], 2),
        "BLEU es-en": round(r["summary"]["bleu"]["es-en"], 2),
        "chrF2 en-es": round(r["summary"]["chrf2"]["en-es"], 2),
        "chrF2 es-en": round(r["summary"]["chrf2"]["es-en"], 2),
        "mean BLEU": round(r["summary"]["mean_bleu"], 2),
    })
display(pd.DataFrame(rows).sort_values("mean BLEU", ascending=False))